# Step 8 : 数値安定化について

## 1: $Q_{uu,n}$の正則性と正定値化

[Step5 backward pass](/notes/step5.ipynb)の入力修正則の導出において、Q関数の二次近似式の$Q_n$を$\delta u_n$で偏微分して、ゼロとなる停留点を最小点の候補であると導出した。

これが最小点となるための十分条件は、$Q_n$の$\delta u_n$によるHessianが正定値であることであった。

$$
\frac{\partial^2 Q_n}{\partial \ \delta u_n^2} = Q_{uu,n} \succ 0
$$

$Q_{uu,n}$は以下のステージコストの$u$によるHessianと、価値関数の状態$X$によるHessianで構成される。

$$
Q_{uu,n} = \ell_{uu,n} + B_n^{T} V_{XX,n+1} B_n
$$

$\ell_{uu,n} \succ 0, V_{XX,n+1} \succeq 0$ であれば、$Q_{uu,n} \succ 0$ となるが、$V_{XX,n+1},\ell_{uu,n}$は非線形な非凸な問題では負の曲率を持つ場合があり、$Q_{uu,n}$が必ずしも正定値であることは通常保証されない。

そこで次のように、$Q_{uu,n}$が正定値となるように、十分な大きさの正則化係数$\mu \ge 0$を選び、以下のように$\tilde{Q}_{uu,n}$を構成する。

$$
\tilde{Q}_{uu,n} = Q_{uu,n} + \mu I
$$

$\tilde{Q}_{uu,n} \succ 0$ であれば、次の入力修正則$\delta u_n$は、正則化後の局所二次Q関数を$\delta X_n$を固定して、入力の摂動$\delta u_n$について一意に最小化する。

$$
\delta u_n = k_n + K_n \delta X_n
$$

$$
k_n = -\tilde{Q}_{uu,n}^{-1} Q_{u,n}, \quad K_n = -\tilde{Q}_{uu,n}^{-1} Q_{uX,n}
$$

### 入力ペナルティ

$Q_n$を$\delta u_n$に関する項でまとめると次のようになる。$C$は$\delta u_n$に依存しない項である。

$$
Q_n(\delta u_n) = (Q_{u,n} + Q_{uX,n} \ \delta X_n)^T \delta u_n + \frac{1}{2}\delta u_n^T \ Q_{uu,n} \ \delta u_n + C 
$$

これに入力ペナルティを加えると、次のようになり、

$$
\begin{aligned}
\tilde{Q}_n &= Q_n(\delta u_n) + \frac{\mu}{2} \delta u_n^T \delta u_n \\
&= (Q_{u,n} + Q_{uX,n} \ \delta X_n)^T \delta u_n + \frac{1}{2}\delta u_n^T \ \left(Q_{uu,n} +  \mu I \right)\ \delta u_n + C 
\end{aligned}
$$

この$\delta u_n$についてのHessianが$\tilde{Q}_{uu,n}$である。

$$
\tilde{Q}_{uu,n} = Q_{uu,n} +  \mu I
$$ 

つまり、$\mu$は$\delta u_n$に対する重みであるため、大きくすると、$\delta u_n$を小さくする効果がある。

### 正則化係数$\mu$の調整方法

前節の通り、$\mu$を大きくすると入力修正則が小さくなるので、軌道の改善が遅くなる。そこで、最初から大きな値ではなく、次のように調整を行う。

$\tilde{Q}_{uu,n}$が正定値行列であるかはCholesky分解の結果より判別することが出来る。

backward passではすでにCholesky分解を用いた$k_n, K_n$の計算と、Cholesky分解を行う$Q_{uu,n}$の対称化処理を行っている。

これに正則化係数$\mu$の調整を加える。

backward passに引数として$\mu$を与える。そしてCholesky分解では対称化した$\tilde{Q}_{uu,n}(\mu)$の分解を行う。もし分解に失敗したら、$\tilde{Q}_{uu,n}(\mu)$ は正定値対称行列ではないとなるため、$\mu$を調整してbackward pass を再計算する。

$\mu$の初期値$\mu_{\text{init} > 0$とし、backward pass を再計算する場合は、$\mu \leftarrow \max(\mu_{\text{init}}, 10 \mu)$ とする。

また、backward pass ではcholesky分解の結果が示すのは$\tilde{Q}_{uu,n}(\mu)$が正定値行列であることであり、実際にコストが下がるかは forward pass で確認する必要がある。

そこで、forward passのline searchでもコストが下がらない場合、$\mu$を増やしてbackward passを再計算する。コストが下がれば、次の反復では$mu = \mu_{\text{init}}$として計算を行う。
